In [7]:
from app.intent import detect_intent
from app.tfidf_search import search

test_questions = [
    {
        "question": "Jak skontaktować się z dziekanatem?",
        "expected_intent": "kontakt",
        "expected_url_contains": "dziekanat"
    },
    {
        "question": "Gdzie znajdę numer telefonu do uczelni?",
        "expected_intent": "kontakt",
        "expected_url_contains": "kontakt"
    },
    {
        "question": "Jak dostać miejsce w akademiku?",
        "expected_intent": "akademik",
        "expected_url_contains": "studenci/akademiki-pb"
    },
    {
        "question": "Gdzie są informacje o domach studenta?",
        "expected_intent": "akademik",
        "expected_url_contains": "studenci/akademiki-pb"
    },
    {
        "question": "Jak wygląda rekrutacja na studia?",
        "expected_intent": "rekrutacja",
        "expected_url_contains": "rekrutacja-krok-po-kroku"
    },
    {
        "question": "Jakie dokumenty są potrzebne przy rekrutacji?",
        "expected_intent": "rekrutacja",
        "expected_url_contains": "rekrutacja-krok-po-kroku"
    },
    {
        "question": "Jakie kierunki studiów oferuje Politechnika Białostocka?",
        "expected_intent": "studia",
        "expected_url_contains": "studia"
    },
    {
        "question": "Gdzie znajdę program studiów informatycznych?",
        "expected_intent": "studia",
        "expected_url_contains": "studia"
    },
    {
        "question": "Jak uzyskać stypendium socjalne?",
        "expected_intent": "stypendia",
        "expected_url_contains": "sekcja-swiadczen-dla-studentow"
    },
    {
        "question": "Jakie są rodzaje stypendiów?",
        "expected_intent": "stypendia",
        "expected_url_contains": "sekcja-swiadczen-dla-studentow"
    }
]

print(f"Liczba pytań testowych: {len(test_questions)}")

Liczba pytań testowych: 10


## Ewaluacja Intencji

In [8]:
correct = 0

for item in test_questions:
    question = item["question"]
    expected = item["expected_intent"]

    predicted = detect_intent(question)

    if predicted == expected:
        correct += 1
        status = "OK"
    else:
        status = "BŁĄD"

    print(
        f"{status:5} | "
        f"expected={expected:12} | "
        f"predicted={predicted:12} | "
        f"{question}"
    )

accuracy = correct / len(test_questions)

print()
print(f"Accuracy intencji = {accuracy:.3f}")
print(f"Accuracy intencji = {accuracy * 100:.1f}%")

OK    | expected=kontakt      | predicted=kontakt      | Jak skontaktować się z dziekanatem?
OK    | expected=kontakt      | predicted=kontakt      | Gdzie znajdę numer telefonu do uczelni?
OK    | expected=akademik     | predicted=akademik     | Jak dostać miejsce w akademiku?
OK    | expected=akademik     | predicted=akademik     | Gdzie są informacje o domach studenta?
OK    | expected=rekrutacja   | predicted=rekrutacja   | Jak wygląda rekrutacja na studia?
OK    | expected=rekrutacja   | predicted=rekrutacja   | Jakie dokumenty są potrzebne przy rekrutacji?
OK    | expected=studia       | predicted=studia       | Jakie kierunki studiów oferuje Politechnika Białostocka?
OK    | expected=studia       | predicted=studia       | Gdzie znajdę program studiów informatycznych?
OK    | expected=stypendia    | predicted=stypendia    | Jak uzyskać stypendium socjalne?
OK    | expected=stypendia    | predicted=stypendia    | Jakie są rodzaje stypendiów?

Accuracy intencji = 1.000
Accuracy in

## Ewaluacja Wyszukiwania (Recall@5)

In [9]:
K = 5
correct = 0

for item in test_questions:
    question = item["question"]
    expected_url = item["expected_url_contains"].lower()

    results = search(question, k=K)

    found = any(
        expected_url in result["url"].lower()
        for result in results
    )

    if found:
        correct += 1
        status = "OK"
    else:
        status = "BŁĄD"

    print(f"{status:5} | {question}")
    print(f"expected url contains: {expected_url}")
    print("returned urls:")

    for result in results:
        print(f" - {result['url']} | score={result['score']:.3f}")

    print()

recall_at_5 = correct / len(test_questions)

print(f"Recall@{K} = {recall_at_5:.3f}")
print(f"Recall@{K} = {recall_at_5 * 100:.1f}%")

BŁĄD  | Jak skontaktować się z dziekanatem?
expected url contains: dziekanat
returned urls:
 - https://pb.edu.pl/kontakt/dane-teleadresowe | score=0.220
 - https://pb.edu.pl/dss/kontakt | score=0.076
 - https://kandydacipb.edu.pl/kontakt | score=0.015

OK    | Gdzie znajdę numer telefonu do uczelni?
expected url contains: kontakt
returned urls:
 - https://pb.edu.pl/kontakt/dane-teleadresowe | score=0.139

OK    | Jak dostać miejsce w akademiku?
expected url contains: studenci/akademiki-pb
returned urls:
 - https://pb.edu.pl/studenci/akademiki-pb | score=0.512

OK    | Gdzie są informacje o domach studenta?
expected url contains: studenci/akademiki-pb
returned urls:
 - https://pb.edu.pl/studenci/akademiki-pb | score=0.459

OK    | Jak wygląda rekrutacja na studia?
expected url contains: rekrutacja-krok-po-kroku
returned urls:
 - https://kandydacipb.edu.pl/studia-i-stopnia/rekrutacja-krok-po-kroku-studia-i-stopnia | score=0.693
 - https://kandydacipb.edu.pl/rekrutacja/studia-ii-stopnia/r